# 1. Import and paths

In [ ]:
#Steps: upload data, split in train/test, pre-process ->feed PANN
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch
import joblib
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from IPython.display import display

In [ ]:
#Set paths for import
ROOT = Path("/home/basti/code/Kitty3000-ML")
DATA_DIR = ROOT / "data" / "CatSound_originals"
MANIFEST_PATH = ROOT / "data" / "manifest.csv"

In [ ]:
# Allow imports from src/ when running inside notebooks/.
sys.path.insert(0, str(ROOT / "src"))
from kitty3000_ml.preprocess import load_clip, N_FFT, HOP_LENGTH

In [ ]:
#Set SR to 32000 as requirement (todo: check details) and seconds to 7 for now (experiment later)
SR = 32000
SECONDS = 7.0

BATCH_SIZE = 1

#model name - experiment with seconds
RUN_NAME = f"panns_cnn14_{SR}hz_{SECONDS:g}s"

#set output directory
OUTPUT_DIR = ROOT / "data" / "processed" / RUN_NAME
MODEL_DIR = ROOT / "models"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the existing manifest.
manifest = pd.read_csv(MANIFEST_PATH, dtype=str)

# Build paths for audio loading.
manifest["full_path"] = manifest["path"].map(lambda p: DATA_DIR / p)

In [ ]:
manifest.head()

# 2. Load PANN model

In [ ]:
from panns_inference import AudioTagging

#Load pre-trained model with default learning weights
pann = AudioTagging(checkpoint_path=None)

#test: select one training data point
sample_path = manifest.loc[
    manifest["split"] == "train", "full_path"
].iloc[0]

#load recording with SR and Seconds, format 32-bit floating
#32000 sampels/second for 7 seconds = 224,000 samples
waveform = load_clip(
    sample_path,
    sr=SR,
    seconds=SECONDS,
).astype(np.float32)

#use inference on waveform
#returns audioset_scores (527 categories) and sample_embeddings /2048) ->input for classifier later
audioset_scores, sample_embedding = pann.inference(waveform[None, :])

print("Waveform:", waveform.shape)              # (224000,)
print("AudioSet scores:", audioset_scores.shape) # (1, 527)
print("Embedding:", sample_embedding.shape)      # (1, 2048)

In [ ]:
#Now that I loaded PANN with all parameters and tried it on one test sample, lets apply it to the entire dataset
#go through each row and create the 2048 features for each audio sample

embeddings = []

#this is just for fun to see what the PANN would classify
predicted_scores = []

#Go through each audio/row, load clip with parameters, infer score and embedding
for path in tqdm(manifest["full_path"], desc="Extracting features"):
    waveform = load_clip(
        path, sr=SR, seconds=SECONDS
    ).astype(np.float32)

    scores, embedding = pann.inference(waveform[None, :])

    embeddings.append(embedding)
    predicted_scores.append(scores[0])

#contains all embeddings in same order as the full_path
X = np.concatenate(embeddings, axis=0)

# One row per recording, one column per AudioSet category.
scores_df = pd.DataFrame(
    predicted_scores,
    columns=pann.labels,
)

# Separate table containing metadata and the highest-scoring prediction.
predictions_df = manifest[
    ["path", "label", "group", "split"]
].reset_index(drop=True).copy()

predictions_df["panns_prediction"] = scores_df.idxmax(axis=1)
predictions_df["panns_score"] = scores_df.max(axis=1)

display(predictions_df.head())

# 3. Train, validate, test 

In [ ]:
y = manifest["label"].to_numpy()

#Split manifest according to the 3 classes train, val, test
train_mask = manifest["split"].eq("train").to_numpy()
val_mask = manifest["split"].eq("val").to_numpy()
test_mask = manifest["split"].eq("test").to_numpy()

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Train:", X_train.shape)
print("Val:  ", X_val.shape)
print("Test: ", X_test.shape)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

results = []
best_score = -1.0

#Create pipeline and run Logistic Regression with 4 different C values
#the smaller C, the highe the regularization ->lets check validation later for optimal C
for C in [0.01, 0.1, 1.0, 10.0]:
    candidate = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2000),
    )

    #fit with training data only
    candidate.fit(X_train, y_train)

    #predict with validation data
    val_pred = candidate.predict(X_val)

    #calculate F1 based on validation set
    score = f1_score(
        y_val, val_pred, average="macro", zero_division=0
    )

    results.append({
        "C": C,
        "val_accuracy": accuracy_score(y_val, val_pred),
        "val_macro_f1": score,
    })

    if score > best_score:
        classifier = candidate
        best_score = score
        best_C = C

display(
    pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
)

print(f"Selected C={best_C} | Validation macro F1={best_score:.3f}")

In [ ]:
#Export the model to model path
MODEL_PATH = MODEL_DIR / f"{RUN_NAME}_classifier.joblib"

joblib.dump({
    "classifier": classifier,
    "sr": SR,
    "seconds": SECONDS,
    "crop_n_fft": N_FFT,
    "crop_hop_length": HOP_LENGTH,
    "panns_checkpoint": "Cnn14_mAP=0.431.pth",
}, MODEL_PATH)

print("Saved:", MODEL_PATH)

In [ ]:
#Final test run
RUN_FINAL_TEST = True

if RUN_FINAL_TEST:
    test_pred = classifier.predict(X_test)

    print(f"Test accuracy: {accuracy_score(y_test, test_pred):.3f}")
    print(
        f"Test macro F1: "
        f"{f1_score(y_test, test_pred, average='macro', zero_division=0):.3f}"
    )

    print(classification_report(
        y_test,
        test_pred,
        labels=classifier.classes_,
        digits=3,
        zero_division=0,
    ))

    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        test_pred,
        labels=classifier.classes_,
        xticks_rotation=90,
        colorbar=False,
        ax=ax,
    )
    plt.tight_layout()
    plt.show()

else:
    print("Finish validation before enabling final test evaluation.")